# 10. Unicode, Regex and the Python String Toolkit

Real-world string work: text from users, logs, APIs. This notebook covers what actual jobs need beyond DSA: **Unicode**, **regular expressions**, and the **built-in string methods** you should know by heart.

**Learning order:** UNICODE AND BYTES → NORMALIZATION → CASE-INSENSITIVE → METHODS → FORMATTING → REGEX → LOG PARSING → INTERVIEW THINKING

## 1. Characters vs bytes

A Python `str` is a sequence of **Unicode code points**. A file or network message is **bytes**. Converting is **encoding** (`str → bytes`) and **decoding** (`bytes → str`), usually with **UTF-8**, which uses 1 to 4 bytes per character.

Consequence: `len(text)` counts **characters**, `len(text.encode())` counts **bytes**. An emoji can be one character but 4 bytes. Databases and APIs often limit **bytes**, not characters.

In [1]:
print(len("a"), len("é"), len("😀"))                                  # 1 1 1
print(len("a".encode()), len("é".encode()), len("😀".encode()))      # 1 2 4
print(ord("A"), chr(97), "é".encode("utf-8"), b"\xc3\xa9".decode("utf-8"))   # 65 a b'\xc3\xa9' é

1 1 1
1 2 4
65 a b'\xc3\xa9' é


## 2. Normalization and case-insensitive comparison

The same visible text can have **different code points**: `é` can be one character (U+00E9) or `e` + a combining accent (U+0301). They look equal but `==` says `False`. Normalize before comparing (`NFC`).

For case-insensitive comparison use **`casefold()`**, not `lower()` (German `ß` folds to `ss`).

In [2]:
import unicodedata

a, b = "\u00e9", "e\u0301"
print(a == b, unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))   # False True
print("Straße".casefold() == "STRASSE".casefold(), "Straße".lower() == "STRASSE".lower())   # True False

False True
True False


## 3. Methods you should know

| Need | Method |
|---|---|
| remove whitespace at the ends | `s.strip()`, `lstrip`, `rstrip` |
| split / join | `s.split(",")`, `s.split()` (any whitespace), `",".join(parts)`, `s.splitlines()` |
| split once | `s.partition("=")` → `(before, sep, after)`, `rsplit(",", 1)` |
| search | `find` (returns -1), `index` (raises), `count`, `startswith`, `endswith` (accept a tuple) |
| replace | `s.replace(old, new, count)` |
| tests | `isdigit`, `isalpha`, `isalnum`, `isspace`, `isupper`, `islower` |
| padding | `zfill(5)`, `ljust`, `rjust`, `center` |
| translate | `str.maketrans("abc", "xyz")` then `s.translate(table)` (fast char mapping/removal) |

In [3]:
s = "  key = value  "
print(s.strip().partition("="))                                   # ('key ', '=', ' value')
print("a,b,,c".split(","), "a b  c".split())                      # ['a', 'b', '', 'c'] ['a', 'b', 'c']
print("file.tar.gz".rsplit(".", 1), "42".zfill(5))                # ['file.tar', 'gz'] 00042
print("hello world".translate(str.maketrans("", "", "lo")))       # he wrd
print("report.pdf".endswith((".pdf", ".docx")))                   # True

('key ', '=', ' value')
['a', 'b', '', 'c'] ['a', 'b', 'c']
['file.tar', 'gz'] 00042
he wrd
True


## 4. Formatting with f-strings

`f"{value:>10.2f}"`: width 10, right aligned, 2 decimals. `f"{n:,}"`: thousands separators. `f"{x!r}"`: `repr`. `f"{name=}"` (3.8+) prints `name='...'`, handy for debugging.

In [4]:
price, qty = 1234.5678, 3
print(f"{price:>10.2f}|{qty:03d}|{1234567:,}".strip())   # 1234.57|003|1,234,567
name = "Manish"
print(f"{name=}", f"{3/7:.3%}")                  # name='Manish' 42.857%

1234.57|003|1,234,567
name='Manish' 42.857%


## 5. Regular expressions (`re`)

A **pattern language** to find, validate and extract text.

| Pattern | Meaning |
|---|---|
| `.` | any character (except newline) |
| `\d`, `\w`, `\s` | digit, word character, whitespace (uppercase = the opposite) |
| `*`, `+`, `?`, `{m,n}` | 0+, 1+, 0 or 1, between m and n repetitions |
| `[abc]`, `[^abc]`, `[a-z]` | a set of characters, anything except them, a range |
| `^`, `$` | start, end of the string |
| `( )`, `(?P<name> )` | capture group, named group |
| `a\|b` | alternation |
| `*?`, `+?` | **non-greedy** (shortest match) |

Functions: `re.search` (first match anywhere), `re.match` (only at the start), `re.fullmatch` (the whole string), `re.findall`, `re.finditer`, `re.sub`, `re.split`, `re.compile` (reuse a pattern).

Use **raw strings** (`r"\d+"`) so backslashes are not interpreted by Python.

In [5]:
import re

text = "Order #123 shipped on 2026-11-18 to a@b.com and c@d.org"
print(re.findall(r"\d+", text))                                           # ['123', '2026', '11', '18']
print(re.search(r"(\d{4})-(\d{2})-(\d{2})", text).groups())               # ('2026', '11', '18')
print(re.findall(r"[\w.]+@[\w.]+\.\w+", text))                            # ['a@b.com', 'c@d.org']
print(re.sub(r"\s+", " ", "too    many   spaces"))                        # too many spaces
print(re.match(r"\d+", "abc123"), re.search(r"\d+", "abc123").group())    # None 123
print(re.findall(r"<.+?>", "<a><b>"), re.findall(r"<.+>", "<a><b>"))     # ['<a>', '<b>'] ['<a><b>']

['123', '2026', '11', '18']
('2026', '11', '18')
['a@b.com', 'c@d.org']
too many spaces
None 123
['<a>', '<b>'] ['<a><b>']


## 6. Mini example: parsing log lines

Named groups make extraction readable. Real systems parse logs, URLs, config files and API responses like this.

In [6]:
LOG = re.compile(r"(?P<level>INFO|WARN|ERROR)\s+\[(?P<time>[\d:]+)\]\s+(?P<msg>.*)")

lines = ["INFO [10:01:22] server started", "ERROR [10:05:09] db timeout", "WARN [10:06:00] slow query"]
errors = [m.group("msg") for l in lines if (m := LOG.match(l)) and m.group("level") == "ERROR"]
print(errors)   # ['db timeout']

['db timeout']


## 7. Interview thinking

**In DSA interviews:** regex is usually **not allowed** when the problem asks you to implement matching (Regular Expression Matching is a DP problem). For parsing tasks and real work, regex and `split`/`partition` are the right tools.

**Say:** "Strings in Python are Unicode, so `len` counts characters. If a limit is in bytes I'd check `len(s.encode())`."

**Common mistakes:** `lower()` for case-insensitive compare (use `casefold`), forgetting raw strings, greedy `.*` swallowing too much, `re.match` when `search` was meant, assuming 1 character = 1 byte, catastrophic backtracking in nested quantifiers (like `(a+)+`).

## Quick Revision

- `str` = Unicode code points, bytes need encode/decode (UTF-8: 1 to 4 bytes).
- Normalize (`NFC`) and `casefold()` before comparing user text.
- `partition`, `split`, `join`, `translate`, `startswith(tuple)`.
- f-string specs: `>10.2f`, `,`, `03d`, `=`.
- `re`: raw strings, `search` vs `match`, groups, non-greedy `*?`.